# PathSnap — HMM + Viterbi Route Reconstruction

**Compatible with your existing preprocessing, feature-engineering and Optuna-tuned Random Forest notebooks.**

This notebook:
1. Loads `outputs/random_forest_tuned/selected_random_forest.joblib` (or supported baseline fallback).
2. Uses the **same 19 scaled features** in `processingdata/{train,val,test,evaluator_test}/model_data.npz`, in their original row order.
3. Reuses `candidates.csv.gz`, `points.csv.gz`, and the original `.nodes/.arcs` road-network files for each trip.
4. Computes HMM emissions from RF candidate probabilities + GPS distance; computes **directed road-graph shortest-path transition costs**, then decodes the most likely road-candidate sequence using **Viterbi**.
5. Exports per-trip outputs, route-agreement metrics and a sample **Folium** map. **No model fitting happens here.**

**Critical evaluation caveats:** `y=on_route` only indicates that an arc belongs somewhere in the reference route. `y_true_arc` is a distance/heading pseudo-label. Neither is exact timestamp-level truth. We report these metrics as *proxy* indicators; sequence overlap with the reference `.route` is also an approximate route-level metric. Do not present them as exact GPS-point matching accuracy.

**Evaluation integrity:** HMM parameters are set in the configuration cell. You may compare configurations using **validation only**; do not tune using student-test or evaluator-test reference routes. The `.route` reference is read only for reporting after decoding, never for selecting candidate roads or computing transitions.

**Runtime:** graph-based Viterbi is substantially more expensive than Random Forest inference, especially on every GPS observation in all splits. For an initial smoke test set `MAX_TRIPS_PER_SPLIT=1` and `MAX_POINTS_PER_TRIP=200`; set both to `None` for full evaluation.


## 1. Install dependencies (once)

In [ ]:
# Uncomment and run this if any package is missing:
# %pip install numpy pandas scipy scikit-learn joblib matplotlib folium


## 2. Configuration and project paths

In [ ]:
import json, time, heapq, warnings
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

try:
    import folium
except ImportError:
    folium = None

SEED = 42
# Set these to None to run on ALL points and ALL trips in each available split.
MAX_TRIPS_PER_SPLIT = None
MAX_POINTS_PER_TRIP = None
RUN_SPLITS = ['Validation', 'Student Test', 'Evaluator Test', 'Train']

# HMM score parameters (calibrate on validation ONLY, if you tune these).
RF_WEIGHT = 1.0        # weight of Random Forest log-probability
GPS_SIGMA_M = 25.0     # Gaussian GPS-to-road distance scale
TRANSITION_BETA_M = 65.0  # route-distance vs GPS-step mismatch scale
MAX_BRIDGE_M = 1500.0  # upper bound for directed graph search between candidates
MAX_SPEED_MPS = 60.0   # reject implausible network travel speed
RESTART_PENALTY = 18.0 # fallback if all road transitions are disconnected
LOG_EPS = 1e-8

cwd = Path.cwd().resolve()
roots = [cwd, cwd/'Hackathon', cwd.parent, cwd.parent/'Hackathon', cwd.parent.parent]
PROJECT_ROOT = next((p for p in roots if (p/'processingdata'/'train'/'model_data.npz').exists()
                     and (p/'Student folder').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate inner Hackathon/ folder containing processingdata/train/model_data.npz and Student folder.')
DATA = PROJECT_ROOT/'processingdata'
RAW = PROJECT_ROOT/'Student folder'
OUT = PROJECT_ROOT/'outputs'/'hmm_viterbi'
OUT.mkdir(parents=True,exist_ok=True)

MODEL_OPTIONS = [
    PROJECT_ROOT/'outputs'/'random_forest_tuned'/'selected_random_forest.joblib',
    PROJECT_ROOT/'outputs'/'random_forest_tuned'/'tuned_random_forest.joblib',
    PROJECT_ROOT/'outputs'/'random_forest_all'/'random_forest_model.joblib',
    PROJECT_ROOT/'outputs'/'random_forest'/'random_forest_model.joblib',
]
MODEL_PATH = next((p for p in MODEL_OPTIONS if p.is_file()),None)
if MODEL_PATH is None:
    raise FileNotFoundError('No trained Random Forest found. Run PathSnap_RandomForest_Tuned_All_Datasets.ipynb through its model-saving cell first. Checked: '+str(MODEL_OPTIONS))
print('Project:', PROJECT_ROOT)
print('RF model:', MODEL_PATH)
print('HMM outputs:', OUT)


## 3. Load saved classifier, features and per-split candidate metadata

In [ ]:
# joblib/pickle files must come only from your own trusted project.
artifact=joblib.load(MODEL_PATH)
if isinstance(artifact,dict):
    rf=artifact.get('model')
    model_feature_names=artifact.get('feature_names')
else:
    rf=artifact
    model_feature_names=None
if rf is None or not hasattr(rf,'predict_proba'):
    raise TypeError('Saved artifact does not contain a classifier with predict_proba')

SPLITS = {'Train':'train','Validation':'val','Student Test':'test','Evaluator Test':'evaluator_test'}
RAW_SPLITS = {'Train':'train','Validation':'train','Student Test':'student_test','Evaluator Test':'evaluator_test'}

def load_split(label):
    path=DATA/SPLITS[label]/'model_data.npz'
    if not path.exists():
        print(f'SKIP {label}: missing {path}')
        return None
    # Only load .npz created by YOUR trusted local feature-engineering notebooks.
    with np.load(path,allow_pickle=True) as z:
        keys=set(z.files)
        required={'X','arc_id','point_idx','groups','feature_names'}
        if not required.issubset(keys):
            raise ValueError(f'{path}: missing {required-keys}')
        d={key:np.asarray(z[key]).copy() for key in keys}
    d['X']=np.asarray(d['X'],dtype=np.float32)
    if not np.isfinite(d['X']).all():
        raise ValueError(f'{path}: X contains NaNs or infinities')
    d['groups']=d['groups'].astype(str)
    d['features']=[str(s) for s in d['feature_names']]
    if model_feature_names is not None and list(model_feature_names)!=d['features']:
        raise ValueError(f'{label}: feature order differs from saved RF model')
    if d['X'].shape[1] != len(d['features']):
        raise ValueError(f'{label}: feature dimensions mismatch')
    folder=DATA/SPLITS[label]
    candidate_path=folder/'candidates.csv.gz'
    points_path=folder/'points.csv.gz'
    if not candidate_path.exists() or not points_path.exists():
        raise FileNotFoundError(f'{label}: required candidates.csv.gz or points.csv.gz not found in {folder}')
    # Row order is preserved by the original feature-engineering notebook.
    cand=pd.read_csv(candidate_path,low_memory=False,dtype={'subject_id':str})
    points=pd.read_csv(points_path,low_memory=False,dtype={'subject_id':str})
    if len(cand)!=len(d['X']):
        raise ValueError(f'{label}: CSV candidate rows != NPZ X rows')
    if not np.array_equal(cand['arc_id'].to_numpy(dtype=int), d['arc_id'].astype(int)):
        raise ValueError(f'{label}: arc_id order in CSV differs from NPZ; STOP to avoid wrong scores')
    if not np.array_equal(cand['point_idx'].to_numpy(dtype=int),d['point_idx'].astype(int)):
        raise ValueError(f'{label}: point_idx order in CSV differs from NPZ; STOP')
    if not np.array_equal(cand['subject_id'].astype(str).to_numpy(),d['groups']):
        raise ValueError(f'{label}: groups order in CSV differs from NPZ; STOP')
    probs=rf.predict_proba(d['X'])[:,list(rf.classes_).index(1)]
    cand=cand.copy()
    cand['rf_probability']=probs
    return {'d':d,'candidates':cand,'points':points,'probabilities':probs}

loaded={}
for label in RUN_SPLITS:
    if label not in SPLITS:
        raise ValueError(f'Unknown split {label}')
    item=load_split(label)
    if item is not None:
        loaded[label]=item
        print(f'{label}: {len(item["candidates"]):,} candidate rows; {item["points"].subject_id.nunique()} trips')
if not loaded:
    raise RuntimeError('No processed datasets were found')


## 4. Load a trip’s directed road graph from the original `.nodes` and `.arcs` files

In [ ]:
EARTH_R=6371008.8

def project(lon,lat,lon0,lat0):
    x=np.radians(lon-lon0)*EARTH_R*np.cos(np.radians(lat0))
    y=np.radians(lat-lat0)*EARTH_R
    return np.column_stack((x,y))

def load_network(split_label,sid):
    sid=str(sid)
    prefix=RAW/RAW_SPLITS[split_label]/sid/sid
    paths=[prefix.with_suffix(ext) for ext in ('.nodes','.arcs')]
    if not all(p.exists() for p in paths):
        raise FileNotFoundError('Missing raw road files for trip '+sid+': '+str(paths))
    nodes=np.loadtxt(paths[0],ndmin=2)
    arcs=np.loadtxt(paths[1],dtype=np.int64,ndmin=2)
    if nodes.shape[1]!=2 or arcs.shape[1]!=2:
        raise ValueError(f'{sid}: unexpected .nodes or .arcs format')
    if np.any(arcs<0) or np.any(arcs>=len(nodes)):
        raise ValueError(f'{sid}: invalid road graph node indexes')
    lon0=float(nodes[:,0].mean()); lat0=float(nodes[:,1].mean())
    xy=project(nodes[:,0],nodes[:,1],lon0,lat0)
    U,V=arcs[:,0],arcs[:,1]
    length=np.linalg.norm(xy[V]-xy[U],axis=1)
    if not np.isfinite(length).all():
        raise ValueError(f'{sid}: invalid road geometry')
    # Match the canonicalization of preprocessing: first arc for a (u,v) pair.
    canonical=pd.Series(np.arange(len(arcs))).groupby([U,V]).transform('min').to_numpy(dtype=int)
    valid=(canonical==np.arange(len(arcs))) & (U!=V) & (length>0)
    adjacency=[[] for _ in range(len(nodes))]
    for arc in np.flatnonzero(valid):
        adjacency[int(U[arc])].append((int(V[arc]),float(length[arc]),int(arc)))
    route_path=prefix.with_suffix('.route')
    route=np.loadtxt(route_path,dtype=int,ndmin=1) if route_path.is_file() else None
    if route is not None and (np.any(route<0) or np.any(route>=len(arcs))):
        raise ValueError(f'{sid}: reference route has invalid arc IDs')
    return {'xy':xy,'arcs':arcs,'U':U,'V':V,'length':length,'adj':adjacency,
            'canonical':canonical,'valid':valid,'route':route,
            'lon0':lon0,'lat0':lat0,'nodes_lonlat':nodes}

# Bounded shortest path. This is directed; a reverse-only road is NOT treated as traversable.
def shortest_to_targets(net,source,targets,max_distance,return_paths=False):
    source=int(source)
    remaining=set(int(t) for t in targets)
    dist={source:0.0}
    parent={}
    heap=[(0.0,source)]
    found={}
    while heap and remaining:
        cost,node=heapq.heappop(heap)
        if cost>max_distance:
            break
        if cost>dist.get(node,float('inf'))+1e-9:
            continue
        if node in remaining:
            found[node]=cost
            remaining.remove(node)
        for nxt,weight,arc in net['adj'][node]:
            nd=cost+weight
            if nd<=max_distance and nd+1e-9<dist.get(nxt,float('inf')):
                dist[nxt]=nd
                if return_paths:
                    parent[nxt]=(node,arc)
                heapq.heappush(heap,(nd,nxt))
    if not return_paths:
        return found
    paths={}
    for dest in found:
        pos=dest; seq=[]
        while pos!=source:
            if pos not in parent:
                break
            prev,arc=parent[pos]; seq.append(arc); pos=prev
        paths[dest]=list(reversed(seq))
    return found,paths


## 5. HMM emissions and directed transition scores

In [ ]:
def emission_log_scores(cand):
    # RF is trained for "somewhere on route", so these are candidate scores,
    # NOT calibrated p(road arc at precise observation). Normalize per GPS point.
    prob=np.clip(cand['rf_probability'].to_numpy(dtype=float),LOG_EPS,1-LOG_EPS)
    d=cand['dist'].to_numpy(dtype=float)
    score=RF_WEIGHT*np.log(prob) - 0.5*(d/GPS_SIGMA_M)**2
    score-=np.max(score)  # numerically stable within each GPS observation
    return score

def transition_log_matrix(prev,curr,net,gps_distance,dt):
    """Entry [i,j] is legal directed travel from previous arc position i to current j."""
    a0=prev['arc_id'].to_numpy(dtype=int); a1=curr['arc_id'].to_numpy(dtype=int)
    f0=prev['frac_along'].to_numpy(dtype=float); f1=curr['frac_along'].to_numpy(dtype=float)
    if np.any(~net['valid'][a0]) or np.any(~net['valid'][a1]):
        raise ValueError('Candidate references an invalid road arc')
    upper=min(MAX_BRIDGE_M,max(gps_distance*3+100,dt*MAX_SPEED_MPS+40,150))
    start_targets=set(int(net['U'][arc]) for arc in a1)
    # A single graph search per distinct previous-arc terminal node.
    costs_by_end={}
    for source in set(int(net['V'][arc]) for arc in a0):
        costs_by_end[source]=shortest_to_targets(net,source,start_targets,upper)
    T=np.full((len(a0),len(a1)),-np.inf,dtype=float)
    for i,arc_a in enumerate(a0):
        for j,arc_b in enumerate(a1):
            if arc_a==arc_b and f1[j]>=f0[i]:
                travel=(f1[j]-f0[i])*net['length'][arc_a]
            else:
                middle=costs_by_end[int(net['V'][arc_a])].get(int(net['U'][arc_b]),float('inf'))
                travel=(1-f0[i])*net['length'][arc_a]+middle+f1[j]*net['length'][arc_b]
            if not np.isfinite(travel) or travel>upper+1e-8:
                continue
            if dt>0 and travel>dt*MAX_SPEED_MPS+40:
                continue
            # Compare road distance against GPS displacement. Penalize long detours.
            T[i,j]=-abs(travel-gps_distance)/TRANSITION_BETA_M-0.0002*travel
    return T


## 6. Viterbi decoder (including explicit gap/disconnection restarts)

In [ ]:
def decode_segment(point_groups,points,net,progress_label=''):
    """point_groups: candidate DataFrame per GPS observation in timestamp order."""
    if not point_groups:
        return [],[],[]
    emissions=[emission_log_scores(g) for g in point_groups]
    scores=emissions[0].copy()
    parents=[None]
    restarted=[np.zeros(len(emissions[0]),dtype=bool)]
    for t in range(1,len(point_groups)):
        prev=point_groups[t-1]; curr=point_groups[t]
        gpsd=float(np.hypot(points[t]['x']-points[t-1]['x'],points[t]['y']-points[t-1]['y']))
        dt=max(0.0,float(points[t]['t']-points[t-1]['t']))
        T=transition_log_matrix(prev,curr,net,gpsd,dt)
        total=scores[:,None]+T
        best_idx=np.argmax(total,axis=0)
        best_score=np.max(total,axis=0)
        restart=~np.isfinite(best_score)
        if np.any(restart):
            # No physically legal directed transition; mark a discontinuity.
            prev_best=int(np.argmax(scores))
            best_idx[restart]=prev_best
            best_score[restart]=scores[prev_best]-RESTART_PENALTY
        scores=emissions[t]+best_score
        if not np.isfinite(scores).any():
            raise RuntimeError('All Viterbi scores became nonfinite')
        scores-=np.max(scores)  # safe re-centering; preserves argmax
        parents.append(best_idx)
        restarted.append(restart)
        if progress_label and t%500==0:
            print(f'  {progress_label}: decoded {t:,}/{len(point_groups):,} GPS observations')
    selected=[0]*len(point_groups)
    selected[-1]=int(np.argmax(scores))
    for t in range(len(point_groups)-1,0,-1):
        selected[t-1]=int(parents[t][selected[t]])
    jumps=[False]+[bool(restarted[t][selected[t]]) for t in range(1,len(point_groups))]
    return selected,jumps,emissions


## 7. Reconstruct a road-arc sequence and measure reference-route overlap

In [ ]:
def ordered_arc_sequence(chosen,net):
    """Insert directed shortest-path bridge arcs between chosen candidates.
    This is an approximate reconstructed arc sequence, not exact turn-by-turn GPS truth.
    """
    if len(chosen)==0:
        return [],0
    out=[int(chosen.iloc[0]['arc_id'])]
    failures=0
    for i in range(1,len(chosen)):
        prev=chosen.iloc[i-1]; cur=chosen.iloc[i]
        arc_a,arc_b=int(prev['arc_id']),int(cur['arc_id'])
        if arc_a==arc_b and float(cur['frac_along'])>=float(prev['frac_along']):
            continue
        if arc_a!=arc_b:
            first=int(net['V'][arc_a]); last=int(net['U'][arc_b])
            result,paths=shortest_to_targets(net,first,[last],MAX_BRIDGE_M,return_paths=True)
            if last in result:
                out.extend(paths[last])
            else:
                failures+=1
            out.append(arc_b)
        else:
            # Same arc but GPS fractional position moved backwards; path may need a loop.
            first=int(net['V'][arc_a]); last=int(net['U'][arc_b])
            result,paths=shortest_to_targets(net,first,[last],MAX_BRIDGE_M,return_paths=True)
            if last in result:
                out.extend(paths[last]);out.append(arc_b)
            else:
                failures+=1
    compact=[out[0]]
    for arc in out[1:]:
        if arc!=compact[-1]:compact.append(int(arc))
    return compact,failures

def route_overlap(predicted,net):
    if net['route'] is None or not len(predicted):
        return {'Route Arc Precision':np.nan,'Route Arc Recall':np.nan,'Route Arc F1':np.nan}
    # Set-based arc coverage, not exact route order or temporal alignment.
    target=set(net['canonical'][net['route']].tolist())
    found=set(net['canonical'][np.asarray(predicted,dtype=int)].tolist())
    shared=len(target & found)
    pr=shared/len(found) if found else 0.0
    re=shared/len(target) if target else 0.0
    f1=2*pr*re/(pr+re) if pr+re else 0.0
    return {'Route Arc Precision':pr,'Route Arc Recall':re,'Route Arc F1':f1}


## 8. Decode one trip — preserve GPS time order, reset across time gaps

In [ ]:
def decode_trip(split_label,sid,split_obj):
    sid=str(sid)
    allc=split_obj['candidates']
    allp=split_obj['points']
    cand=allc[allc.subject_id.astype(str)==sid].copy()
    pts=allp[allp.subject_id.astype(str)==sid].copy()
    if cand.empty or pts.empty:
        raise ValueError(f'{sid}: missing candidates or points')
    net=load_network(split_label,sid)
    pts=pts.sort_values('t',kind='stable').reset_index(drop=True)
    if MAX_POINTS_PER_TRIP is not None:
        pts=pts.iloc[:MAX_POINTS_PER_TRIP].copy()
    allowed=set(pts.point_idx.astype(int).tolist())
    cand=cand[cand.point_idx.isin(allowed)].copy()
    cand_by_point={int(idx):part.reset_index(drop=True)
                   for idx,part in cand.groupby('point_idx',sort=False)}
    all_selected=[]
    all_jumps=[]
    segment_sequences=[]
    for seg,segpts in pts.groupby('seg_id',sort=False):
        groups=[]; positions=[]
        for _,pt in segpts.iterrows():
            current=cand_by_point.get(int(pt.point_idx))
            if current is None or current.empty:
                raise ValueError(f'{sid}: GPS observation {pt.point_idx} lacks road candidates')
            groups.append(current.reset_index(drop=True))
            positions.append(pt)
        picks,jumps,_=decode_segment(groups,positions,net,progress_label=f'{sid}/segment{seg}')
        rows=[]
        for current,idx in zip(groups,picks):
            rows.append(current.iloc[int(idx)].to_dict())
        chosen=pd.DataFrame(rows)
        chosen['viterbi_restart']=jumps
        all_selected.append(chosen)
        all_jumps.extend(jumps)
        segment_sequences.append(chosen)
    selected=pd.concat(all_selected,ignore_index=True)
    # Greedy RF baseline: highest RF probability for each GPS observation
    greedy=(cand.sort_values(['point_idx','rf_probability'],ascending=[True,False])
            .drop_duplicates('point_idx').set_index('point_idx'))
    selected['greedy_arc_id']=selected['point_idx'].map(greedy['arc_id']).to_numpy()
    selected['greedy_probability']=selected['point_idx'].map(greedy['rf_probability']).to_numpy()
    summary={
        'Split':split_label, 'Trip':sid,'GPS Points':len(selected),
        'Viterbi Restarts':int(sum(all_jumps)),
        'Viterbi Restart Rate':float(np.mean(all_jumps)),
        'HMM On-Route Rate':float(selected['on_route'].mean()) if 'on_route' in selected else np.nan,
        'RF Top1 On-Route Rate':float(greedy.loc[selected.point_idx,'on_route'].mean()) if 'on_route' in greedy else np.nan,
        'HMM Pseudo True Arc Rate':float(selected['is_true_arc'].mean()) if 'is_true_arc' in selected else np.nan,
        'RF Pseudo True Arc Rate':float(greedy.loc[selected.point_idx,'is_true_arc'].mean()) if 'is_true_arc' in greedy else np.nan,
    }
    route=[]; bridge_failures=0
    for chosen in segment_sequences:
        arcs,failures=ordered_arc_sequence(chosen,net)
        route.extend(arcs); bridge_failures+=failures
    summary.update(route_overlap(route,net))
    summary['Route Sequence Arcs']=len(route)
    summary['Missing Graph Bridges']=bridge_failures
    return summary,selected,route,net,pts


## 9. Run all available train / validation / student-test / evaluator trips

In [ ]:
reports=[]
visual_example=None
start=time.perf_counter()
for split_label,obj in loaded.items():
    sids=sorted(obj['points'].subject_id.astype(str).unique())
    if MAX_TRIPS_PER_SPLIT is not None:
        sids=sids[:MAX_TRIPS_PER_SPLIT]
    print(f'\n=== {split_label}: {len(sids)} trip(s) ===')
    for n,sid in enumerate(sids,1):
        begin=time.perf_counter()
        summary,chosen,route,net,pts=decode_trip(split_label,sid,obj)
        summary['Decode Seconds']=time.perf_counter()-begin
        reports.append(summary)
        dirname=OUT/split_label.lower().replace(' ','_')
        dirname.mkdir(exist_ok=True,parents=True)
        chosen.to_csv(dirname/f'{sid}_viterbi_candidates.csv',index=False)
        pd.DataFrame({'arc_id':route}).to_csv(dirname/f'{sid}_reconstructed_route.csv',index=False)
        if visual_example is None and split_label in ('Validation','Student Test'):
            visual_example=(sid,split_label,chosen,route,net,pts)
        print(f'[{n}/{len(sids)}] {sid}: GPS={summary["GPS Points"]:,} '
              f'RF on-route={summary["RF Top1 On-Route Rate"]:.2%} '
              f'HMM on-route={summary["HMM On-Route Rate"]:.2%} '
              f'restarts={summary["Viterbi Restarts"]} '
              f'time={summary["Decode Seconds"]:.1f}s')

report_df=pd.DataFrame(reports)
report_df.to_csv(OUT/'per_trip_route_metrics.csv',index=False)
print(f'\nDone in {time.perf_counter()-start:.1f}s')
display(report_df)


## 10. Per-split summary and comparison plots

In [ ]:
if report_df.empty:
    raise ValueError('No trips decoded')
metric_cols=['GPS Points','HMM On-Route Rate','RF Top1 On-Route Rate',
             'HMM Pseudo True Arc Rate','RF Pseudo True Arc Rate',
             'Route Arc Precision','Route Arc Recall','Route Arc F1',
             'Viterbi Restart Rate','Missing Graph Bridges','Decode Seconds']
summary_table=report_df.groupby('Split')[metric_cols].mean().round(4)
summary_table.insert(0,'Trips',report_df.groupby('Split').size())
summary_table.to_csv(OUT/'per_split_route_metrics.csv')
print('PER-SPLIT METRICS (UNWEIGHTED TRIP MEAN)')
display(summary_table)

cols=['RF Top1 On-Route Rate','HMM On-Route Rate']
ax=summary_table[cols].plot(kind='bar',figsize=(9,5),ylim=(0,1))
ax.set_title('RF greedy vs HMM/Viterbi — on-route proxy metric')
ax.set_ylabel('Mean fraction of selected candidates on reference route')
plt.xticks(rotation=20,ha='right');plt.tight_layout()
plt.savefig(OUT/'rf_vs_hmm_route_proxy.png',dpi=160)
plt.show();plt.close()

print('Reminder: on-route / pseudo true-arc are proxy labels; not exact observation-level accuracy.')


## 11. Folium map — GPS track, selected road segments, reconstructed path

In [ ]:
if folium is None:
    print('Install folium with %pip install folium and rerun this cell.')
elif visual_example is None:
    print('No validation or student-test trip decoded for map. Set RUN_SPLITS accordingly.')
else:
    sid,split_label,chosen,route,net,pts=visual_example
    # Convert road projected XY back to longitude/latitude using same per-trip frame.
    def xy_to_lonlat(xy):
        lon=net['lon0']+np.degrees(xy[:,0]/(EARTH_R*np.cos(np.radians(net['lat0']))))
        lat=net['lat0']+np.degrees(xy[:,1]/EARTH_R)
        return np.column_stack((lat,lon))
    gps_ll=xy_to_lonlat(pts[['x','y']].to_numpy())
    m=folium.Map(location=list(gps_ll[len(gps_ll)//2]),zoom_start=14,tiles='OpenStreetMap')
    folium.PolyLine(gps_ll.tolist(),color='red',weight=2,opacity=.6,tooltip='Raw cleaned GPS trajectory').add_to(m)
    # Full reconstructed arc sequence; draw each road arc. A gap is not a verified connection.
    for arc in route:
        u,v=net['arcs'][int(arc)]
        a,b=net['nodes_lonlat'][int(u)],net['nodes_lonlat'][int(v)]
        folium.PolyLine([[a[1],a[0]],[b[1],b[0]]],color='blue',weight=4,opacity=.7).add_to(m)
    for _,r in chosen.iloc[::max(1,len(chosen)//150)].iterrows():
        arc=int(r['arc_id']); u,v=net['arcs'][arc]
        xy=net['xy'][u]+float(r['frac_along'])*(net['xy'][v]-net['xy'][u])
        ll=xy_to_lonlat(np.array([xy]))[0]
        folium.CircleMarker(ll,radius=2,color='green',fill=True,
                            tooltip=f'Viterbi arc={arc}, RF={r["rf_probability"]:.3f}').add_to(m)
    folium.LayerControl().add_to(m)
    map_path=OUT/f'{split_label.lower().replace(" ","_")}_{sid}_route_map.html'
    m.save(str(map_path))
    print('Saved interactive Folium map:',map_path)
    display(m)


## 12. Save settings and interpret results

In [ ]:
config={
    'model_file':str(MODEL_PATH), 'model_selection':'fixed trained model; no retraining',
    'splits_run':list(loaded),'max_trips_per_split':MAX_TRIPS_PER_SPLIT,
    'max_points_per_trip':MAX_POINTS_PER_TRIP,
    'rf_weight':RF_WEIGHT,'gps_sigma_m':GPS_SIGMA_M,
    'transition_beta_m':TRANSITION_BETA_M,'max_bridge_m':MAX_BRIDGE_M,
    'max_speed_mps':MAX_SPEED_MPS,'restart_penalty':RESTART_PENALTY,
    'labels':'on_route is trip-level; is_true_arc is pseudo observation-level; route overlap is set-based',
    'restrictions':'Do not tune parameters on evaluator_test or student_test reference labels',
}
(OUT/'hmm_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
print('All results saved under:',OUT)
print('Key files: per_trip_route_metrics.csv, per_split_route_metrics.csv, *_route_map.html')
print('Measure actual reconstruction quality with reference route order and connectivity; these proxies are not absolute accuracy.')
